# C-MAPSS FD001 — RUL Benchmark on Kaggle 2×T4

Full 20 631-row training set, 4 chains × 150 sweeps, auto-pmap across both T4 GPUs.

Every chunk (~30 sweeps) is checkpointed to `examples/c_mapss/results/inference/FD001/` — so a Kaggle session death costs you at most the last chunk, not the whole run.

**Measured throughput on 2×T4** (from a prior interrupted attempt): ~40 s per chain-sweep post-compile. With 4 chains (2 per device via `fori_loop` serialization) × 150 sweeps, the projected wallclock is **~3.3 hours**. Comfortably under Kaggle's 12 h weekly GPU quota.

## Kaggle setup

1. Settings → Accelerator → **GPU T4 ×2**
2. Settings → Internet → **on**
3. Run cells **sequentially, not "Run All"**. The smoke test (cell 4) has to pass before you commit to cell 6.

## If the kernel dies mid-run

Re-open the notebook, run cells 1-3 (fast), skip 4-5, and re-run cell 6 as-is. The `--resume` flag picks up from the last checkpoint — you only lose the sweeps since the last 30-sweep chunk.

In [ ]:
# 1. Clone repo + install no-deps (preserves Kaggle's JAX+CUDA stack)
import os

WORKDIR = "/kaggle/working/jaxcross"
BRANCH = "feat/cmapss-pdm-example"  # change to 'main' after merge

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git fetch && git reset --hard)
os.chdir(WORKDIR)
!git fetch origin && (git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}) && git pull origin {BRANCH}

%pip install -e . --no-deps -q
%pip install polars scikit-learn -q

print(f"Branch: {BRANCH}")
print(f"CWD: {os.getcwd()}")

In [ ]:
# 2. Verify both GPUs are visible
import jax

print("JAX:", jax.__version__)
print("backend:", jax.default_backend())
print("devices:", jax.devices())
n_dev = jax.device_count()
print("device_count:", n_dev)
assert n_dev == 2, f"Expected 2 T4 GPUs, got {n_dev} — check accelerator settings"
!nvidia-smi --query-gpu=name,memory.free,memory.total --format=csv

In [ ]:
# 3. Fetch + preprocess FD001 (~20 s total)
!python examples/c_mapss/fetch_cmapss.py
!python examples/c_mapss/preprocess_cmapss.py FD001

In [ ]:
# 4. SMOKE TEST — ~3-5 min end-to-end pipeline validation
#    2 chains x 6 sweeps x 1000 rows. If this fails, something is wrong in
#    preprocessing / GPU / library plumbing. Fix before committing to cell 6.
!PYTHONUNBUFFERED=1 python examples/c_mapss/run_inference.py FD001 --smoke

In [ ]:
# 5. Clear the smoke-test checkpoint before the real run
#    (smoke ran on 1000 rows; the full run needs data_shape=(20631, 20))
import shutil
from pathlib import Path

p = Path("examples/c_mapss/results/inference/FD001")
if p.exists():
    shutil.rmtree(p)
    print(f"Cleared {p}")

In [ ]:
# 6. Full inference — 4 chains x 150 sweeps x 20 631 rows, pmap across 2 T4s.
#    Checkpoints every 30 sweeps (5 chunks total).
#
#    If the kernel dies, re-run this exact cell — --resume picks up from the
#    last checkpoint without redoing already-finished sweeps.
#
#    Expected wallclock: ~3.3 h. ETA prints from the 2nd chunk onwards.
!PYTHONUNBUFFERED=1 python examples/c_mapss/run_inference.py FD001 \
    --chains 4 --sweeps 150 --diag-every 30 \
    --max-views 16 --max-clusters 32 --seed 42 \
    --resume

In [ ]:
# 7. Convergence check — Gelman-Rubin R-hat + ESS + trace plateau
import jax.numpy as jnp
import numpy as np

from crosscat.diagnostics import effective_sample_size, gelman_rubin_rhat

tr = jnp.asarray(np.load("examples/c_mapss/results/inference/FD001/log_joint_traces.npy"))
rhat = float(gelman_rubin_rhat(tr))
ess = float(effective_sample_size(tr))
q = max(1, tr.shape[1] // 4)
last = tr[:, -q:]
swing = float((last.max() - last.min()) / jnp.abs(tr.mean()))
print(f"Trace shape: {tr.shape}  (chains, diagnostic_points)")
print(f"Rhat = {rhat:.4f}   ESS = {ess:.1f}   last-quartile swing = {swing:.2%}")
print(f"Final log_joints: {[float(x) for x in tr[:, -1]]}")

if rhat < 1.10 and swing < 0.01:
    print("\nCONVERGED: 150 sweeps was sufficient.")
elif rhat < 1.20:
    print(
        "\nAPPROX CONVERGED: fine for production; re-run cell 6 with --sweeps 250 --resume for tighter R-hat."
    )
else:
    print("\nNOT CONVERGED: re-run cell 6 with --sweeps 300 --resume (keeps current progress).")

In [ ]:
# 8. Evaluate RUL via 4-chain BMA + 90/95/99% CI coverage
!PYTHONUNBUFFERED=1 python examples/c_mapss/evaluate_rul.py FD001 --samples 1000

In [ ]:
# 9. Best-chain-only (for the BMA-vs-single comparison)
!PYTHONUNBUFFERED=1 python examples/c_mapss/evaluate_best_chain.py FD001 --samples 1000

In [ ]:
# 10. Same-data sklearn baselines (Ridge + RandomForest)
!python examples/c_mapss/baseline_rul.py FD001

In [ ]:
# 11. Final leaderboard vs published baselines
import json
from pathlib import Path

EVAL = Path("examples/c_mapss/results/evaluation/FD001")
BASE = Path("examples/c_mapss/results/baselines/FD001")
INF = Path("examples/c_mapss/results/inference/FD001")

bma = json.loads((EVAL / "metrics.json").read_text())
best = json.loads((EVAL / "best_chain_metrics.json").read_text())
baselines = json.loads((BASE / "baseline_metrics.json").read_text())
meta = json.loads((INF / "inference_meta.json").read_text())

print(
    f"Inference: {meta['n_chains']} chains x {meta['n_sweeps']} sweeps, "
    f"{meta['data_shape'][0]} rows, {meta['elapsed_seconds'] / 60:.1f} min on {meta['mode']}"
)
print(f"Final log-joints: {meta['final_log_joints']}")
print()
print(f"{'Model':45s}  {'MAE':>6s}  {'RMSE':>6s}  {'R^2':>6s}")
print("-" * 70)
print(f"{'Transformer (2024-25, published)':45s}  {11.90:>6.2f}  {'   -':>6s}  {'   -':>6s}")
print(f"{'CNN-LSTM, Li 2018 (published)':45s}  {12.61:>6.2f}  {'   -':>6s}  {'   -':>6s}")
print(f"{'LSTM, Zheng 2017 (published)':45s}  {13.52:>6.2f}  {'   -':>6s}  {'   -':>6s}")
print(
    f"{'RandomForest (same training rows)':45s}  {baselines['random_forest']['mae']:>6.2f}  "
    f"{baselines['random_forest']['rmse']:>6.2f}  {baselines['random_forest']['r2']:>6.3f}"
)
print(
    f"{'Ridge (same training rows)':45s}  {baselines['ridge']['mae']:>6.2f}  "
    f"{baselines['ridge']['rmse']:>6.2f}  {baselines['ridge']['r2']:>6.3f}"
)
print(
    f"{'jaxcross BMA (all chains)':45s}  {bma['bma']['mae']:>6.2f}  "
    f"{bma['bma']['rmse']:>6.2f}  {bma['bma']['r2']:>6.3f}"
)
print(
    f"{'jaxcross best-chain-only':45s}  {best['point']['mae']:>6.2f}  "
    f"{best['point']['rmse']:>6.2f}  {best['point']['r2']:>6.3f}"
)
print()
print("CI coverage (nominal vs empirical):")
for level in (90, 95, 99):
    bma_c = bma[f"ci_{level}"]
    best_c = best[f"ci_{level}"]
    print(
        f"  {level}% CI:  BMA={bma_c['coverage']:.1%} (w={bma_c['avg_width']:.1f})  "
        f"best={best_c['coverage']:.1%} (w={best_c['avg_width']:.1f})"
    )

## Runbook / troubleshooting

| Symptom | Action |
|---|---|
| Cell 2 asserts `device_count != 2` | Settings → Accelerator → **GPU T4 ×2**, then restart kernel. |
| Smoke test (cell 4) hangs > 8 min | Check `!nvidia-smi` — if both GPUs are idle, the JAX install is broken; restart kernel and re-run cell 1. |
| Full run (cell 6) kernel dies | Re-run cell 6 as-is. `--resume` loads the last checkpoint; you lose ≤ 30 sweeps. |
| Full run first chunk > 90 min | Check `!nvidia-smi`. If GPU util is 100 %, compile is still running; wait. If idle, interrupt and restart cell 6. |
| Cell 7 reports R-hat > 1.20 | Re-run cell 6 with `--sweeps 300 --resume` — adds 150 more sweeps on top of what's there. |

## Expected runtime budget (2×T4)

| Cell | Task | Time |
|---|---|---:|
| 1 | Clone + install | ~30 s |
| 2 | GPU verify | ~5 s |
| 3 | Fetch + preprocess | ~20 s |
| 4 | **Smoke test** | ~3-5 min |
| 5 | Clear smoke cache | <1 s |
| 6 | **Full inference** (4 chains × 150 sweeps × 20 631 rows) | **~3.3 h** |
| 7 | Convergence check | ~1 s |
| 8 | Evaluate + BMA | ~2-5 min |
| 9 | Best-chain eval | ~1 min |
| 10 | sklearn baselines | ~1 min |
| 11 | Leaderboard | <1 s |
| **Total** | | **~3.5 h** |

Well within the 12 h weekly free-tier budget, with room for ~2 retries if a kernel dies.

## Output artifacts

Everything lives under `/kaggle/working/jaxcross/examples/c_mapss/results/` and downloads with the notebook:

- `inference/FD001/chain_{0..3}.jxc` — per-chain packed states (latest checkpoint)
- `inference/FD001/best_chain.jxc` — highest-log-joint chain
- `inference/FD001/log_joint_traces.npy` — convergence trace
- `inference/FD001/inference_meta.json` — config + timings + `last_completed_sweep`
- `inference/FD001/train_used.npy` — exact training rows
- `evaluation/FD001/metrics.json, best_chain_metrics.json, rul_predictions.csv, rul_predictions.arrow`
- `baselines/FD001/baseline_metrics.json`